# RAG Module · Day 23 — **Context Window Management**

**Phase 1 · Module 3: Prompt Engineering & RAG Architecture** · ~2 hrs

Retrieval hands you chunks; the model has a **fixed token budget**. A 50-page policy PDF plus chat history plus the system prompt won't fit — and even inside a 200K window, stuffing everything *degrades* answers (lost-in-the-middle) and costs tokens you pay for. This morning you profiled **bytes**; this afternoon, **tokens**.

Four strategies, and when each wins:
1. **Token budgeting** — count first, decide what fits.
2. **Compression** (LLMLingua-style) — drop low-information tokens, keep meaning.
3. **Sliding window / map-reduce** — process a long doc in pieces.
4. Measure the **accuracy vs compression** trade-off — how far can you compress before answers break?

> Pure Python, stdlib. No keys. Token counting is word-based here; production uses the model's real tokenizer (`tiktoken` / the Anthropic counter). The strategies are identical.

## 1. Count tokens, set a budget

You can't manage what you don't measure. A rough token count (≈ ¾ of a word, but word-count is fine for reasoning) tells you whether the assembled prompt fits. The budget is split: system + query + history + **retrieved context** ≤ window.

In [1]:
import re, math

def count_tokens(text):
    return len(re.findall(r"\S+", text))          # word ≈ token proxy

# a 50-"page" policy doc: 50 sections, each a few sentences
SECTIONS = []
for p in range(50):
    SECTIONS.append(
        f"Section {p+1}. Please note that, generally speaking, the applicable policy for product "
        f"number {p+1} states that the overdraft limit for this account is {(p+1)*100} pounds "
        f"and that the monthly fee is {p % 5} pounds respectively, as amended from time to time."
    )
DOC = "\n".join(SECTIONS)

WINDOW = 1000
SYSTEM, QUERY = "You are a banking policy assistant.", "What is the overdraft limit for product 7?"
reserved = count_tokens(SYSTEM) + count_tokens(QUERY) + 150   # + room for the answer
budget_for_context = WINDOW - reserved

print("doc tokens         :", count_tokens(DOC))
print("window             :", WINDOW)
print("reserved (sys+q+ans):", reserved)
print("budget for context :", budget_for_context)
print("=> full doc fits?  :", count_tokens(DOC) <= budget_for_context)

doc tokens         : 2000
window             : 1000
reserved (sys+q+ans): 164
budget for context : 836
=> full doc fits?  : False


☝ The doc is several times the context budget — you **cannot** pass it whole. Every strategy below answers the same question: *given a fixed budget, which tokens make the cut?* Naive RAG's answer (top-k chunks) is one option; the others recover more of the doc per token.

## 2. Compression — drop low-information tokens

Prose is padded with **stopwords and filler** (*"please note that, generally speaking, ... respectively, as amended from time to time"*) that carry little signal for retrieval or answering. **LLMLingua** does exactly this with a small model scoring token informativeness; we approximate by dropping stopwords/filler and keeping content tokens — the same idea, typically **2–3× fewer tokens** at near-equal answer quality.

In [2]:
STOP = set("a an the of to and or for is are be been being this that these those with without "
           "please note generally speaking respectively as from time amended applicable states "
           "number this account it its they them please by at on in has have had will shall may "
           "product per each any all such other than then so we you your our".split())
FILLER = re.compile(r"\b(please note that|generally speaking|as amended from time to time|"
                    r"respectively|from time to time|it should be noted)\b", re.I)

def compress(text, keep_numbers=True):
    text = FILLER.sub("", text)
    out = []
    for w in text.split():
        core = re.sub(r"[^\w]", "", w).lower()
        if keep_numbers and re.search(r"\d", w):
            out.append(w)                      # never drop numbers/codes
        elif core and core not in STOP:
            out.append(w)
    return " ".join(out)

orig = SECTIONS[6]
comp = compress(orig)
print("original :", orig)
print("\ncompressed:", comp)
print(f"\ntokens {count_tokens(orig)} -> {count_tokens(comp)}  "
      f"({count_tokens(orig)/max(count_tokens(comp),1):.1f}x)")

original : Section 7. Please note that, generally speaking, the applicable policy for product number 7 states that the overdraft limit for this account is 700 pounds and that the monthly fee is 1 pounds respectively, as amended from time to time.

compressed: Section 7. policy 7 overdraft limit 700 pounds monthly fee 1 pounds

tokens 40 -> 12  (3.3x)


☝ The compressed section keeps *Section 7, overdraft limit, 700 pounds, monthly fee* — every fact needed to answer — while shedding the boilerplate, at ~2–3× fewer tokens. Rule: **never compress out numbers, codes, or names** (the `keep_numbers` guard) — those are exactly what a banking query asks for.

## 3. Sliding window / map-reduce over a long doc

Compression alone may still overflow. For a genuinely long doc, **process it in windows**: retrieve/answer over each slice (*map*), then combine (*reduce*). Here we retrieve the most relevant compressed sections until the budget fills — recovering far more of the doc than raw top-k would at the same token cost.

In [3]:
def tok(s): return set(re.findall(r"[a-z0-9]+", s.lower()))

def budgeted_context(query, sections, budget, do_compress):
    q = tok(query)
    ranked = sorted(sections, key=lambda s: len(q & tok(s)), reverse=True)
    picked, used = [], 0
    for s in ranked:
        piece = compress(s) if do_compress else s
        t = count_tokens(piece)
        if used + t > budget: break
        picked.append(piece); used += t
    return picked, used

for do_c in (False, True):
    ctx, used = budgeted_context(QUERY, SECTIONS, budget_for_context, do_c)
    label = "compressed" if do_c else "raw"
    print(f"{label:10}: fit {len(ctx):2d} sections in {used} tokens (budget {budget_for_context})")

raw       : fit 20 sections in 800 tokens (budget 836)
compressed: fit 50 sections in 600 tokens (budget 836)


☝ Same token budget, but compression fits **more sections** into it — more of the document reaches the model per token spent. That's the lever: compression doesn't just save cost, it raises *recall under a fixed window*. Map-reduce extends this further — summarise each window, then answer over the summaries when even the compressed doc overflows.

## 4. The accuracy vs compression trade-off

Compression isn't free — squeeze too hard and you drop the token the answer needs. Sweep the aggressiveness and watch answer accuracy: the goal is the **knee** — maximum compression that still answers correctly.

In [4]:
QUESTIONS = [(f"overdraft limit for product {p}", str(p*100)) for p in range(1, 26)]

def answer_from(context_text, question):
    # extract the limit for the product named in the question
    m = re.search(r"product (\d+)", question)
    if not m: return None
    p = m.group(1)
    # find "product <p> ... <number> pounds" in the assembled context
    hit = re.search(rf"product number {p}\b.*?(\d+) pounds", context_text)
    if not hit:
        hit = re.search(rf"\b{p}\b.*?(\d+) pounds", context_text)
    return hit.group(1) if hit else None

def accuracy(drop_stop):
    correct = 0
    for q, gold in QUESTIONS:
        secs = [(compress(s) if drop_stop else s) for s in SECTIONS]
        ctx = " ".join(secs)                        # whole doc, optionally compressed
        got = answer_from(ctx, q)
        correct += (got == gold)
    return correct / len(QUESTIONS)

raw_tokens  = count_tokens(" ".join(SECTIONS))
comp_tokens = count_tokens(" ".join(compress(s) for s in SECTIONS))
print(f"no compression : acc {accuracy(False):.0%}  | {raw_tokens} tokens")
print(f"compression on : acc {accuracy(True):.0%}  | {comp_tokens} tokens "
      f"({raw_tokens/comp_tokens:.1f}x smaller)")

no compression : acc 100%  | 2000 tokens
compression on : acc 100%  | 600 tokens (3.3x smaller)


☝ Compression cuts the doc several-fold **while accuracy holds** — because the guard kept every `product N` and `NNN pounds`. That's the trade you tune for real workloads: push compression up until the accuracy curve bends, then back off one step. Measure it per corpus — legal text tolerates less compression than marketing copy.

## 5. Recap — tokens are a budget, spend them well

| Strategy | Mechanism | Use when |
|---|---|---|
| **Token budgeting** | count sys+query+history+context ≤ window | always — the precondition |
| **Compression** | drop stopwords/filler, keep numbers/names | context overflows; cut cost + lost-in-the-middle |
| **Sliding window / map-reduce** | process slices, then combine | doc >> window even compressed |
| **Measure trade-off** | accuracy vs compression ratio | tune per corpus — find the knee |

The discipline: **count before you stuff, compress what's padding, never drop a fact, and prove accuracy held.** 